In [ ]:
import sys
import os
sys.path.append(os.path.abspath('..'))


In [ ]:
from src.chunker import *
from src.config import *
from src.data_loader import *
from src.evaluator import *
from src.generator import *
from src.retriever import *
from src.train_model import *
from src.utils import *

In [ ]:
input_excel = OUTPUT_PATH / "dataset_with_rag_answers.xlsx"

In [ ]:
model_name = 'Qwen/Qwen2.5-32B-Instruct'
tokenizer = AutoTokenizer.from_pretrained(model_name)

        
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,
    device_map="auto",
    torch_dtype=torch.bfloat16,
).eval()

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

In [ ]:
import pandas as pd
import numpy as np
import torch
from transformers import pipeline
from langchain_huggingface import HuggingFacePipeline

eval_pipeline = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
    max_new_tokens=1024,
    do_sample=False,            
    repetition_penalty=1.05,
    pad_token_id=tokenizer.eos_token_id,
    return_full_text=False
)
llm_judge = HuggingFacePipeline(pipeline=eval_pipeline)
evaluator = RAGEvaluator(llm_judge=llm_judge, tokenizer=tokenizer)



df = pd.read_excel(input_excel)


required = ['question', 'answer', 'RAG_answer', 'RAG_context']
for col in required:
    if col not in df.columns:
        raise ValueError(f"В Excel отсутствует колонка '{col}'")

print(f"Загружено {len(df)} примеров. Начинаем оценку...")


eval_metrics = []

for idx, row in df.iterrows():
    question = row['question']
    ground_truth = row['answer']
    rag_answer = row['RAG_answer']
    context_str = row['RAG_context']   
    
    try:
        scores = evaluator.evaluate(
            question=question,
            answer=rag_answer,
            context_docs=[],            
            context_str=context_str,
            ground_truth=ground_truth
        )
        eval_metrics.append(scores)
        print(f"[{idx+1}/{len(df)}] Faith: {scores['faithfulness']['score']:.2f} | "
              f"Rel: {scores['answer_relevancy']['score']:.2f} | "
              f"Prec: {scores['context_precision']['score']:.2f} | "
              f"Rec: {scores['context_recall']['score']:.2f}")
    except Exception as e:
        print(f"Ошибка в примере {idx+1}: {e}")

        eval_metrics.append({
            "faithfulness": {"score": 0.0},
            "answer_relevancy": {"score": 0.0},
            "context_precision": {"score": 0.0},
            "context_recall": {"score": 0.0}
        })

for metric in ['faithfulness', 'answer_relevancy', 'context_precision', 'context_recall']:
    df[f'eval_{metric}'] = [m[metric]['score'] for m in eval_metrics]

output_excel = OUTPUT_PATH / 'dataset_with_rag_answers_and_metrics.xlsx'
df.to_excel(output_excel, index=False)


faith = [m['faithfulness']['score'] for m in eval_metrics]
rel   = [m['answer_relevancy']['score'] for m in eval_metrics]
prec  = [m['context_precision']['score'] for m in eval_metrics]
rec   = [m['context_recall']['score'] for m in eval_metrics]

print("\n" + "="*60)
print("СРЕДНИЕ МЕТРИКИ ПО ДАТАСЕТУ")
print("="*60)
print(f"Faithfulness     : {np.mean(faith):.4f}")
print(f"Answer Relevancy : {np.mean(rel):.4f}")
print(f"Context Precision: {np.mean(prec):.4f}")
print(f"Context Recall   : {np.mean(rec):.4f}")
print(f"Q (интегральная) : {np.mean([np.mean(faith), np.mean(rel), np.mean(prec), np.mean(rec)]):.4f}")
print("="*60)

print(f"\nРезультат сохранён в {output_excel}")